# 00. Data Inventory — Dunnhumby

## Mục tiêu

> Xác định bảng có cấu trúc gì và một dòng đại diện cho cái gì.

Notebook này chỉ kiểm tra cấu trúc: shape, physical dtype, semantic role, required columns, entity/time metadata, key cardinality, candidate grain và preview.

Mỗi bảng đi qua cùng một flow: `load → validate contract → build inventory → kiểm tra grain → chốt row meaning`. Phần quan hệ giữa các bảng chưa nằm trong phạm vi notebook này.

| Bảng | Trạng thái | Row meaning hiện tại |
|---|---|---|
| `transaction_data` | Đã triển khai | Một product line trong một basket |
| `product` | Đã triển khai | Một product master record |
| `hh_demographic` | Chờ duyệt | Chưa chốt |
| `campaign_desc` | Chờ duyệt | Chưa chốt |
| `campaign_table` | Chờ duyệt | Chưa chốt |
| `coupon` | Chờ duyệt | Chưa chốt |
| `coupon_redempt` | Chờ duyệt | Chưa chốt |
| `causal_data` | Chờ duyệt | Chưa chốt |

In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import polars as pl

from propensity.eda import build_table_inventory
from propensity.io import scan_csv
from propensity.schema.dunnhumby import (
    PRODUCT_SPEC,
    TRANSACTION_SCHEMA_OVERRIDES,
    TRANSACTION_SPEC,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

## A. `transaction_data`

### A1. Load và validate structural contract

In [ ]:
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "Dunnhumby"
transaction_path = data_dir / "transaction_data.csv"
transactions = scan_csv(
    transaction_path,
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
TRANSACTION_SPEC.validate(transactions)
transactions

`scan_csv()` chỉ tạo lazy query plan. Các phép collect cần thiết nằm trong từng inventory function, không nằm trong notebook.

### A2. Build inventory

In [ ]:
transaction_inventory = build_table_inventory(transactions, TRANSACTION_SPEC)

### A3. Table overview

In [ ]:
pl.DataFrame([asdict(transaction_inventory.overview)])

### A4. Column structure và semantic roles

`physical_dtype` mô tả cách Polars lưu dữ liệu. `semantic_role` mô tả ý nghĩa nghiệp vụ do `DatasetSpec` cung cấp.

In [ ]:
# Required columns đã được validate ở bước 1, không cần hiển thị lại.
transaction_inventory.columns.drop("is_required")

### A5. Entity và ID cardinality

Time column không nằm trong bảng này vì `DAY` là time axis, không phải key.

In [ ]:
transaction_inventory.keys

### A6. Candidate grain

**Grain** là mức chi tiết nhỏ nhất của một bảng, trả lời câu hỏi: *một dòng dữ liệu đại diện cho điều gì?*

Với bảng transaction, candidate grain là `BASKET_ID + PRODUCT_ID`:

- `BASKET_ID` xác định một giỏ hàng/giao dịch.
- `PRODUCT_ID` xác định sản phẩm xuất hiện trong giỏ hàng đó.
- Nếu tổ hợp này unique, mỗi dòng được diễn giải là **một sản phẩm (product line) trong một basket**. `QUANTITY` cho biết số lượng sản phẩm trên dòng, nên không cần một dòng riêng cho từng đơn vị sản phẩm.
- `household_key`, `STORE_ID` và `DAY` mô tả bối cảnh của product line, nhưng không thuộc candidate grain vì một household/store/day có thể xuất hiện ở nhiều dòng.

`is_unique=True` là bằng chứng cấu trúc ủng hộ candidate grain. Nếu `is_unique=False`, có nhiều dòng cùng `BASKET_ID + PRODUCT_ID`; khi đó cần kiểm tra duplicate hoặc tìm thêm cột phân biệt trước khi chốt ý nghĩa mỗi dòng.

In [ ]:
transaction_inventory.grain

### A7. Preview

In [ ]:
transaction_inventory.preview

### A8. Nhận xét cấu trúc

- Entity của bảng là `household_key`.
- Time axis là `DAY`, được lưu dưới dạng ordinal integer với `time_grain=day_index`; notebook không chuyển nó thành calendar date.
- `BASKET_ID`, `PRODUCT_ID` và `STORE_ID` có physical dtype numeric nhưng semantic role là ID.
- Candidate grain là `BASKET_ID + PRODUCT_ID`. Nếu grain report có `is_unique=True`, một dòng được diễn giải là **một product line trong một basket**. Nếu `is_unique=False`, cần xem lại candidate grain hoặc dữ liệu trùng trước khi chốt row meaning.

## B. `product`

Bảng `product` là dimension table mô tả product master. Bảng không có entity column hoặc time column; cấu trúc được kiểm tra bằng product identifiers và candidate grain.

In [ ]:
product_path = data_dir / "product.csv"
products = scan_csv(product_path)
PRODUCT_SPEC.validate(products)
products

In [ ]:
product_inventory = build_table_inventory(products, PRODUCT_SPEC)

### B1. Table overview

In [ ]:
pl.DataFrame([asdict(product_inventory.overview)])

### B2. Column structure và semantic roles

`PRODUCT_ID` và `MANUFACTURER` được khai báo là ID. Các cột phân cấp/mô tả sản phẩm còn lại được suy luận là categorical từ physical dtype string.

In [ ]:
product_inventory.columns.drop("is_required")

### B3. ID cardinality

In [ ]:
product_inventory.keys

### B4. Candidate grain và row meaning

Candidate grain của bảng là `PRODUCT_ID`. Nếu `PRODUCT_ID` unique, mỗi dòng đại diện cho **một product master record** cùng các thuộc tính manufacturer, department, brand, commodity, sub-commodity và current size.

`MANUFACTURER` là ID nhưng không thuộc grain: một manufacturer có thể sở hữu nhiều product.

In [ ]:
product_inventory.grain

### B5. Preview

In [ ]:
product_inventory.preview

### B6. Nhận xét cấu trúc

- Bảng không có entity/time axis; inventory và quality vẫn dùng được nhờ `DatasetSpec` cho phép hai metadata này optional.
- `PRODUCT_ID` là candidate grain và phải unique để bảng đóng vai trò product dimension.
- `MANUFACTURER` có physical dtype numeric nhưng semantic role là ID; không tính mean/std cho cột này.
- Các cột hierarchy và mô tả sản phẩm thuộc semantic role categorical.